## Installations


In [ ]:
# 1. Install Unsloth (The core library for fast VLM loading)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install the necessary backend tools for 4-bit quantization and data handling
!pip install -q --no-deps bitsandbytes accelerate datasets xformers

## Inference Script


In [ ]:
%%writefile run_inference.py
import os
import argparse
import json
import re
import gc
import torch
import pandas as pd
from tqdm import tqdm
import datasets
from datasets import load_dataset
from unsloth import FastVisionModel
from PIL import Image

# 1. Hardware Isolation
parser = argparse.ArgumentParser()
parser.add_argument('--gpu', type=str, required=True)
parser.add_argument('--start', type=int, required=True)
parser.add_argument('--end', type=int, required=True)
args = parser.parse_args()

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu

# 2. Model Initialization
print(f"[GPU {args.gpu}] Loading model...")
model, processor = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)

# 3. Dataset Loading
print(f"[GPU {args.gpu}] Loading dataset...")
test_ds = load_dataset("UkrainianCatholicUniversity/rukopys", data_dir="test", split="train")
test_ds = test_ds.cast_column("image", datasets.Image(decode=False))

actual_end = min(args.end, len(test_ds))
subset = test_ds.select(range(args.start, actual_end))

ALLOWED_TYPES = {"annotation", "formula", "graph", "handwritten", "image", "printed", "table"}

def extract_and_clean_json(text):
    """Extracts JSON, fixes truncation, and enforces Kaggle types."""
    try:
        match = re.search(r'(\[.*\])', text, re.DOTALL)
        if not match: return "[]"
        json_str = match.group(1)
        
        # Parse (with fallback for truncation)
        try:
            regions = json.loads(json_str)
        except json.JSONDecodeError:
            last_brace_idx = json_str.rfind('}')
            if last_brace_idx != -1:
                try:
                    regions = json.loads(json_str[:last_brace_idx + 1] + ']')
                except: return "[]"
            else: return "[]"

        # Enforce Kaggle vocabulary
        for region in regions:
            ctype = region.get('type', '').lower().strip()
            if ctype not in ALLOWED_TYPES:
                if "signature" in ctype: region['type'] = "handwritten"
                elif "stamp" in ctype or "logo" in ctype: region['type'] = "image"
                elif "text" in ctype: region['type'] = "printed"
                else: region['type'] = "annotation"
                
        return json.dumps(regions)
    except: return "[]"

system_prompt = (
    "You are a precise OCR system for Ukrainian documents. "
    "Break the document down into individual lines or distinct regions. "
    "For EVERY distinct line, output an object with exactly three keys:\n"
    "1. 'bbox': [x1, y1, x2, y2]\n"
    "2. 'type': One of [handwritten, printed, formula, table, annotation, image, graph]\n"
    "3. 'text': The exact transcribed text.\n"
    "Output ONLY a valid JSON list."
)
example_json = '[{"bbox": [50, 100, 850, 150], "type": "handwritten", "text": "Перший рядок."}]'

# 4. Inference Loop
submission_data = []
print(f"[GPU {args.gpu}] Starting inference on images {args.start} to {actual_end}...")

for i, item in enumerate(tqdm(subset, position=int(args.gpu))):
    # Grab the real UUID filename directly from the path!
    file_name = os.path.basename(item['image']['path'])
    
    # Now decode the image manually for the model
    image = Image.open(item['image']['path']).convert("RGB")
    image.thumbnail((768, 768), Image.Resampling.LANCZOS) 
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": "Extract all distinct lines/regions from this image."}]},
        {"role": "assistant", "content": [{"type": "text", "text": f"Understood. Format:\n{example_json}"}]},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Now extract all distinct lines/regions from this document."}]}
    ]
    
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(image, input_text, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=3000, 
            temperature=0.0, 
            do_sample=False,
            use_cache=True
        )
    
    prediction = processor.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    clean_regions = extract_and_clean_json(prediction)
    submission_data.append({"image": file_name, "regions": clean_regions})

    # Memory Safety Valve
    if (i + 1) % 5 == 0:
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

# 5. Save Chunk
df = pd.DataFrame(submission_data)
output_filename = f"submission_part_{args.gpu}.csv"
df.to_csv(output_filename, index=False)
print(f"[GPU {args.gpu}] Finished! Saved to {output_filename}")

## 2GPU Orchestrator


In [ ]:
import subprocess
import time
import pandas as pd

print("Launching strictly isolated Dual-GPU inference...")

# Define dataset split
total_images = 387
midpoint = total_images // 2

# Launch Process 1 (Strictly bound to Physical GPU 0)
p0 = subprocess.Popen([
    "python", 
    "run_inference.py", 
    "--gpu", "0", 
    "--start", "0", 
    "--end", str(midpoint)
])

# Stagger the launches to prevent CPU RAM overflow during model loading
print("Waiting 30 seconds for GPU 0 to initialize before launching GPU 1...")
time.sleep(30)

# Launch Process 2 (Strictly bound to Physical GPU 1)
p1 = subprocess.Popen([
    "python", 
    "run_inference.py", 
    "--gpu", "1", 
    "--start", str(midpoint), 
    "--end", str(total_images)
])

print("Both processes are running. Monitoring...")

# Block notebook execution until both scripts finish
p0.wait()
p1.wait()

print("\nProcessing complete. Merging results...")

# Merge the partial CSVs
try:
    df0 = pd.read_csv("submission_part_0.csv")
    df1 = pd.read_csv("submission_part_1.csv")
    final_df = pd.concat([df0, df1], ignore_index=True)

    final_df.to_csv("submission.csv", index=False)
    print(f"Success! Final submission.csv created with {len(final_df)} images.")

except FileNotFoundError:
    print("Error: One or both of the partial CSV files were not found.")